In [2]:
import requests
import torch.nn as nn
import torch
from transformers import GPT2TokenizerFast


url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text

print(f"Dataset length: {len(text):,} characters")



Dataset length: 1,115,394 characters


In [2]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
encoded_text = tokenizer.encode(text)

Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


In [3]:
from torch.utils.data import Dataset, DataLoader

class Dataset:
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.output_ids = []

        txt_ids = tokenizer.encode(txt)

        for i in range(0, len(txt_ids) - max_length, stride):
            input_chunk = txt_ids[i:i+max_length]
            target_chunk = txt_ids[i+1 : i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.output_ids.append(torch.tensor(target_chunk, dtype=torch.long))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.output_ids[idx]

def create_dataloader(txt, batch_size=16, max_length=16, stride=8, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
    dataset = Dataset(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)


In [4]:
dataloader = create_dataloader(text)
first_batch = next(iter(dataloader))

Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


In [5]:
first_batch[0].type()  # torch.LongTensor

'torch.LongTensor'

In [ ]:
class vanillaLSTMCell(nn.Module):
    def __init__(self, vocab_size, emb_size,  hidden_size):
        super(vanillaLSTMCell, self).__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.W_x = nn.Linear(emb_size, hidden_size)
        self.W_z = nn.Linear(2 * hidden_size, hidden_size) # [w_z, r_z] + b_z
        self.W_i = nn.Linear(2 * hidden_size, hidden_size) # [w_i, r_i] + b_i
        self.W_f = nn.Linear(2 * hidden_size, hidden_size) # [w_f, r_f] + b_f
        self.W_o = nn.Linear(2 * hidden_size, hidden_size) # [w_o, r_o] + b_o
        self.vocab_projection = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h_t_minus_one, c_t_minus_one):
        
        _, seq_length = x.shape

        outputs = []
        for t in range(seq_length):
            x_t = self.embedding(x[:, t])
            x_t = self.W_x(x_t)
            
            o_t = torch.sigmoid(self.W_o(torch.cat((h_t_minus_one, x_t), dim=1)))
            f_t = torch.sigmoid(self.W_f(torch.cat((h_t_minus_one, x_t), dim=1)))
            i_t = torch.sigmoid(self.W_i(torch.cat((h_t_minus_one, x_t), dim=1)))
            z_t = torch.tanh(self.W_z(torch.cat((h_t_minus_one, x_t), dim=1)))
            c_t = f_t * c_t_minus_one + i_t * z_t
            h_t = o_t * torch.tanh(c_t)

            c_t_minus_one = c_t
            h_t_minus_one = h_t

            outputs.append(h_t.unsqueeze(1))
        
        outputs = torch.cat(outputs, dim=1)
        logits = self.vocab_projection(outputs)

        return logits, (c_t, h_t)



In [ ]:
class sLSTMCell(nn.Module):
    def __init__(self, vocab_size, emb_size,  hidden_size):
        super(sLSTMCell, self).__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.W_o = nn.Linear(hidden_size + emb_size, hidden_size) # [W_o, R_o] + b_o
        self.W_f = nn.Linear(hidden_size + emb_size, hidden_size) # [W_f, R_f] + b_f
        self.W_i = nn.Linear(hidden_size + emb_size, hidden_size) # [W_i, R_i] + b_i
        self.W_z = nn.Linear(hidden_size + emb_size, hidden_size) # [W_z, R_z] + b_z
        self.vocab_projection = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h_t_minus_one, c_t_minus_one, n_t_minus_one):
        _, seq_length = x.shape

        outputs = []
        for t in range(seq_length):
            x_t = self.embedding(x[:, t])
            
            hidden_cat_input = torch.cat((h_t_minus_one, x_t), dim=1)
            o_t = torch.sigmoid(self.W_o(hidden_cat_input)) # output gate(37)
            f_t = torch.sigmoid(self.W_f(hidden_cat_input)) # forget gate(36)
            #f_t = torch.exp(self.W_f(torch.cat((h_t_minus_one, x_t), dim=1)))
            i_t = torch.exp(self.W_i(hidden_cat_input))     # input gate(35)
            z_t = torch.tanh(self.W_z(hidden_cat_input))    # cell input(34)
            n_t = f_t * n_t_minus_one + i_t                 # normalizer state(32)
            c_t = f_t * c_t_minus_one + i_t * z_t           # cell state(31)
            h_t = o_t * (c_t * (1 / n_t))                   # hidden state(33)

            n_t_minus_one = n_t
            c_t_minus_one = c_t
            h_t_minus_one = h_t

            outputs.append(h_t.unsqueeze(1))
        
        outputs = torch.cat(outputs, dim=1)
        logits = self.vocab_projection(outputs)

        return logits, (c_t, h_t, n_t)

In [ ]:
class sLSTMCell(nn.Module):
    """
    sLSTM cell with stabilized exponential gating and recurrent connections.
    Equations (49)-(53) from the paper.
    """

    def __init__(self, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        # Recurrent connections for memory mixing within the head
        # These are intrinsically block-diagonal globally because they are instantiated per-head
        self.R_i = nn.Linear(hidden_size, hidden_size, bias=False)
        self.R_f = nn.Linear(hidden_size, hidden_size, bias=False)
        self.R_z = nn.Linear(hidden_size, hidden_size, bias=False)
        self.R_o = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, z_pre, o_pre, i_pre, f_pre, state):
        # state: (h, c, n, m)
        h_prev, c_prev, n_prev, m_prev = state

        # Add recurrent hidden-hidden connections
        i_tilde = i_pre + self.R_i(h_prev)
        f_tilde = f_pre + self.R_f(h_prev)
        z_tilde = z_pre + self.R_z(h_prev)
        o_tilde = o_pre + self.R_o(h_prev)

        # (34) cell input
        z_t = torch.tanh(z_tilde)

        # (37) output gate
        o_t = torch.sigmoid(o_tilde)

        # (49) stabilizer state
        m_t = torch.max(f_tilde + m_prev, i_tilde)

        # (50) stabilized input gate
        i_s = torch.exp(i_tilde - m_t)

        # (51) stabilized forget gate
        f_s = torch.exp(f_tilde + m_prev - m_t)

        # (52) stabilized cell state update
        c_t = f_s * c_prev + i_s * z_t

        # (53) stabilized normalizer state update
        n_t = f_s * n_prev + i_s

        # (33) hidden state
        h_t = o_t * (c_t / n_t)

        return h_t, (h_t, c_t, n_t, m_t)

In [ ]:
# sLSTMBlock

class sLSTMBlock(nn.Module):
    def __init__(self, dim, num_heads, vocab_size):
        super(sLSTMBlock, self).__init__()

        self.num_heads = num_heads
        self.dim = dim 
        self.head_dim = dim // num_heads 

        self.embeddings = nn.Embedding(vocab_size, dim)
        self.ln = nn.LayerNorm(dim)
        self.conv4 = nn.Conv1d(dim, dim, 4, padding=3, groups=dim)
        self.swish = nn.SiLU()

        self.i_proj = nn.Conv1d(dim, dim, kernel_size=1, groups=num_heads)
        self.f_proj = nn.Conv1d(dim, dim, kernel_size=1, groups=num_heads)
        self.z_proj = nn.Conv1d(dim, dim, kernel_size=1, groups=num_heads)
        self.o_proj = nn.Conv1d(dim, dim, kernel_size=1, groups=num_heads)

        self.cells = nn.ModuleList(
            [sLSTMCell(self.head_dim) for _ in range(num_heads)]
        )

        self.gn = nn.GroupNorm(num_heads, dim)

        pf_dim = (dim * 4/3)

        self.up_proj = nn.Linear(dim, pf_dim)
        self.gate_proj = nn.Linear(dim, pf_dim)
        self.gelu = nn.GELU()
        self.down_proj = nn.Linear(pf_dim, dim)
        
    def forward(self, x):
        # (batch_size, seq_len, emb_dim)
        x = self.embeddings(x)
        identity = x 
        x_norm = self.ln(x)
        x_T = x.transpose(1, 2)
        x_conv = self.conv4(x_T)
        x_conv = torch.transpose(x_conv, (1,2))[:, :, :-3]
        x_conv = self.swish(x_conv)

        i_pre = self.i_proj(x_conv).transpose(1,2)
        f_pre = self.f_proj(x_conv).transpose(1,2)
        z_pre = self.z_proj(x_norm)
        o_pre = self.o_proj(x_norm)

        batch_size, seq_len, _ = x.shape

        # (head_dim, batch_size, seq_len, num_heads)
        i_pre_heads = i_pre.chunk(self.num_heads, dim=-1) 
        f_pre_heads = f_pre.chunk(self.num_heads, dim=-1)
        z_pre_heads = z_pre.chunk(self.num_heads, dim=-1)
        o_pre_heads = o_pre.chunk(self.num_heads, dim=-1)

        head_outputs = []

        for h in self.num_heads:
            cell = self.cells[h]

            h_t = torch.zeros(batch_size, self.head_dim, device=x.device)
            c_t = torch.zeros(batch_size, self.head_dim, device=x.device)
            n_t = torch.ones(batch_size, self.head_dim, device=x.device)
            m_t = torch.zeros(batch_size, self.head_dim, device=x.device)

            state = (h_t, c_t, n_t, m_t)

            head_h = []

            for t in range(seq_len): 
                h_t, state = cell(
                    z_pre[h][:, t, :],
                    o_pre[h][:, t, :],
                    i_pre[h][:, t, :],
                    f_pre[h][:, t, :],
                    state
                )
                head_h.append(h_t) # h_t size -> (batch_size, 1, head_size)
            
            head_outputs.append(torch.cat(head_h, dim=1)) # (batch_size, seq_len, head_size)
        
        h_out = torch.cat(head_outputs, dim=-1) #(batch_size, seq_len, embedding_size)
        h_out_t = h_out.transpose(1,2)
        h_out_norm = self.gn(h_out_t)
        h_out_norm = h_out_norm.transpose(1,2)

        x = identity + h_out_norm 
        x_norm = self.ln(x)

        up = self.up_proj(x_norm)
        gate = self.gelu(self.gate_proj(x_norm))
        mlp_out = self.down_proj(up * gate)

        return x + mlp_out

In [ ]:
class mLSTMCell(nn.Module):
    """
    mLSTM cell with matrix memory.
    Equations (16)-(18) from the paper.
    """

    def __init__(self, head_dim):
        super().__init__()
        self.head_dim = head_dim

    def forward(self, q, k, v, i_tilde, f_tilde, state):
        # state: (C_prev, n_prev)
        # C_prev: (batch, head_dim, head_dim)
        # n_prev: (batch, head_dim)
        C_prev, n_prev = state

        # Exponential gating
        i_t = torch.exp(i_tilde)  # (batch, 1)
        f_t = torch.exp(f_tilde)  # (batch, 1)

        # (20) Key scaling factor
        k = k / (self.head_dim ** 0.5)

        # (16) Matrix cell state update
        # v: (batch, head_dim), k: (batch, head_dim)
        C_t = f_t.unsqueeze(-1) * C_prev + i_t.unsqueeze(-1) * \
            torch.bmm(v.unsqueeze(2), k.unsqueeze(1))

        # (17) Normalizer state update
        n_t = f_t * n_prev + i_t * k

        # (18) Hidden state pre-activation
        # h_tilde = (C_t @ q) / max(|n_t.T @ q|, 1)
        dot_product = torch.sum(n_t * q, dim=-1, keepdim=True)  # (batch, 1)
        h_tilde = torch.bmm(C_t, q.unsqueeze(2)).squeeze(
            2) / torch.max(torch.abs(dot_product), torch.ones_like(dot_product))

        return h_tilde, (C_t, n_t)

In [ ]:
class mLSTMBlock(nn.Module):
    """
    mLSTM Block as shown in Figure 8.
    Pre up-projection residual structure.
    """

    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads

        # Pre up-projection with projection factor PF=2
        self.PF = 2
        self.hidden_dim = dim * self.PF
        self.head_dim = self.hidden_dim // num_heads

        self.ln = nn.LayerNorm(dim)

        # Up-projection once for the externalized output gate and once for the mLSTM cells
        self.up_proj_gate = nn.Linear(dim, self.hidden_dim)
        self.up_proj_cell = nn.Linear(dim, self.hidden_dim)

        # Causal convolution (kernel size 4) applied dimension-wise
        self.conv1d = nn.Conv1d(self.hidden_dim, self.hidden_dim,
                                kernel_size=4, padding=3, groups=self.hidden_dim)

        # Block-diagonal Projections (implemented efficiently via Conv1d with groups)
        # q and k are projected block-diagonally from hidden_dim -> hidden_dim
        self.q_proj = nn.Conv1d(
            self.hidden_dim, self.hidden_dim, kernel_size=1, groups=num_heads)
        self.k_proj = nn.Conv1d(
            self.hidden_dim, self.hidden_dim, kernel_size=1, groups=num_heads)

        # i and f gates are scalar per head, projected block-diagonally from hidden_dim -> num_heads
        self.i_proj = nn.Conv1d(
            self.hidden_dim, num_heads, kernel_size=1, groups=num_heads)
        self.f_proj = nn.Conv1d(
            self.hidden_dim, num_heads, kernel_size=1, groups=num_heads)

        self.cells = nn.ModuleList([
            mLSTMCell(self.head_dim) for _ in range(num_heads)
        ])

        self.gn = nn.GroupNorm(num_heads, self.hidden_dim)

        # Learnable skip connection from the post-convolution input
        self.learnable_skip = nn.Parameter(torch.ones(self.hidden_dim))

        # Final down-projection
        self.down_proj = nn.Linear(self.hidden_dim, dim)

    def forward(self, x):
        # x: (batch, seq_len, dim)
        residual = x

        # 1. Pre LayerNorm
        x_norm = self.ln(x)

        # 2. Gate branch up-projection and activation (Swish)
        gate_pre = self.up_proj_gate(x_norm)
        gate_act = F.silu(gate_pre)  # (batch, seq_len, hidden_dim)

        # 3. Cell branch up-projection
        cell_in = self.up_proj_cell(x_norm)  # (batch, seq_len, hidden_dim)

        # 4. Values (v) skip the convolution and projections (fed directly)
        v = cell_in

        # 5. Causal Convolution Path
        conv_in = cell_in.transpose(1, 2)  # (batch, hidden_dim, seq_len)
        # Remove padding to maintain causal property
        conv_in = self.conv1d(conv_in)[:, :, :-3]
        conv_in = F.silu(conv_in)  # Swish activation

        # Block-diagonal projections for q, k, i, f
        q = self.q_proj(conv_in).transpose(
            1, 2)  # (batch, seq_len, hidden_dim)
        k = self.k_proj(conv_in).transpose(1, 2)
        
        i_tilde = self.i_proj(conv_in).transpose(
            1, 2)  # (batch, seq_len, num_heads)
        f_tilde = self.f_proj(conv_in).transpose(1, 2)

        # 6. mLSTM Cells (sequential over time)
        batch_size, seq_len, _ = x.shape

        q_heads = q.chunk(self.num_heads, dim=-1)
        k_heads = k.chunk(self.num_heads, dim=-1)
        v_heads = v.chunk(self.num_heads, dim=-1)

        head_outputs = []
        for h in range(self.num_heads):
            cell = self.cells[h]
            C_t = torch.zeros(batch_size, self.head_dim,
                              self.head_dim, device=x.device)
            n_t = torch.zeros(batch_size, self.head_dim, device=x.device)
            state = (C_t, n_t)

            head_h = []
            for t in range(seq_len):
                h_t, state = cell(
                    q_heads[h][:, t, :],
                    k_heads[h][:, t, :],
                    v_heads[h][:, t, :],
                    i_tilde[:, t, h:h+1],
                    f_tilde[:, t, h:h+1],
                    state
                )
                head_h.append(h_t.unsqueeze(1))

            head_outputs.append(torch.cat(head_h, dim=1))

        h_all = torch.cat(head_outputs, dim=-1)

        # 7. GroupNorm
        h_norm = h_all.transpose(1, 2)
        h_norm = self.gn(h_norm)
        h_norm = h_norm.transpose(1, 2)

        # 8. Learnable Skip Connection and Gating
        skip = conv_in.transpose(1, 2) * self.learnable_skip
        h_combined = h_norm + skip
        h_gated = h_combined * gate_act

        # 9. Down-projection and residual connection
        out = self.down_proj(h_gated)

        return residual + out